KPIs.
### 1. TOTAL SALES:
Total money collected from customers for all products sold.

### 2. PROFIT PER PRODUCT:
Selling price minus the cost of the item.

### 3. TOP SELLING CATEGORY:
Which group (e.g. Electronics) brings the most money.

### 4. BASKET SIZE:
Average number of items bought per single transaction.

### 5. RETURN RATE:
Percentage of total sales that were returned by customers.

### 6. OUT-OF-STOCK COUNT:
Count of unique products with zero units in stock.

### 7. STORE PERFORMANCE:
Ranking of the 5 stores based on daily sales volume.

### 8. DISCOUNT IMPACT:
Money lost due to differences in marked vs. sold price.

### 9. REPEAT CUSTOMER COUNT:
Number of unique users with more than one purchase.

### 10. SLOW-MOVING INVENTORY:
Products with zero sales recorded in the last 30 days.

In [0]:
USE SCHEMA data_mart;

-- KPI 1
CREATE OR REPLACE VIEW gold.total_sales AS
SELECT
  sum(total_amount) as total_sales
FROM
  fact_sales

In [0]:
--- KPI 2
-- Profit Per Product
CREATE OR REPLACE VIEW gold.profit_per_product AS
SELECT
    p.product_id,
    p.item_name_description AS product_name,
    p.category_type,
    IFNULL(SUM(fs.profit_amount), 0) AS total_profit
FROM data_mart.dim_products p
LEFT JOIN fact_sales fs USING (product_id)
GROUP BY 1, 2, 3;

In [0]:
-- KPI 3
CREATE OR REPLACE VIEW gold.profit_per_category AS
SELECT
    category_type,
    SUM(total_profit) AS total_profit,
    ROW_NUMBER() OVER (ORDER BY SUM(total_profit) DESC) AS rank
FROM gold.profit_per_product
GROUP BY category_type
ORDER BY total_profit DESC;

In [0]:
-- KPI 4
-- Bucket Size
CREATE OR REPLACE VIEW gold.average_bucket_size AS
select
  AVG(qty_sold) as bucket_size
from
  fact_sales;

In [0]:
-- KPI 5
-- Return Rate
CREATE OR REPLACE VIEW gold.return_rate AS
select
  ROUND(
    (SELECT COUNT(*) FROM fact_returns) / COUNT(*), 2
) AS return_rate
from
  fact_sales;

In [0]:
-- KPI 6
-- Out of stock
create or replace view gold.out_of_stock as
(
  select
    product_id,
    store_id,
    dim_stores.location_name_address as store_name,
    dim_products.item_name_description as product_name
  from
    fact_inv_levels
    left join dim_products using (product_id)
    left join dim_stores using (store_id)
  where
    stock_on_hand = 0
  group by
    product_id,
    store_id,
    dim_stores.location_name_address,
    dim_products.item_name_description
);

In [0]:
-- KPI 7
-- STORE PERFORMANCE
CREATE OR REPLACE VIEW gold.store_performance as
(
  select
    store_id,
    dim_stores.location_name_address as store_name,
    round(avg(qty_sold), 2) as daily_volume,
    row_number() over (order by avg(qty_sold) desc) as rank
  from
    fact_sales
    left join dim_stores using (store_id)
  group by
    store_id, store_name
  order by
    daily_volume desc
)

In [0]:
-- KPI 8
---  DISCOUNT IMPACT
create or replace view gold.total_discount_impact as
(
  select
    sum(discounted_amount) as total_discount
  from
    fact_sales
);

create or replace view gold.total_discount_per_store as
(
  select
    store_id,
    dim_stores.location_name_address as store_name,
    sum(discounted_amount)
  from
    fact_sales
    left join dim_stores using (store_id)
  group by
    store_id, dim_stores.location_name_address
)


In [0]:
-- KPI 9
-- Returning Customers
CREATE OR REPLACE VIEW gold.total_returning_customers AS
SELECT
  COUNT(DISTINCT cust_ref_id) AS returning_customers
FROM
  (
    SELECT
      cust_ref_id
    FROM
      fact_sales
    WHERE
      cust_ref_id != 'UNKNOWN'
    GROUP BY
      cust_ref_id
    HAVING
      COUNT(*) > 1
  );

CREATE OR REPLACE VIEW gold.returning_customer_per_store AS
SELECT
  store_id,
  store_name,
  COUNT(cust_ref_id) AS returning_customers
FROM
  (
    SELECT
      s.store_id,
      ds.location_name_address AS store_name,
      s.cust_ref_id
    FROM
      fact_sales s
        JOIN dim_stores ds
          ON s.store_id = ds.store_id
    WHERE
      s.cust_ref_id != 'UNKNOWN'
    GROUP BY
      1,
      2,
      3
    HAVING
      COUNT(*) > 1
  )
GROUP BY
  1,
  2;

In [0]:
-- kpi 10
-- slow moving inventory
create or replace view gold.slow_moving_inventory as
with products_last_30_days as (
  select
    product_id,
    sale_date
  from
    fact_sales
  where
    DATEDIFF(current_date(), sale_date) < 60
  order by
    sale_date desc
),
sold_products as (
  select distinct
    product_id,
    sale_date
  from
    products_last_30_days
),
slow_moving_inventory as (
  select
    product_id,
    dim_products.item_name_description as product_name,
    dim_products.category_type as category
  from
    fact_inv_levels as inv
    left join sold_products using (product_id)
    left join dim_products using (product_id)
  where
    inv.product_id not in (select product_id from sold_products)
)
select
  *
from
  slow_moving_inventory;